# Whitmore Diagnostics Notebook

This notebook is intentionally lightweight: it loads the pipeline outputs from `taa_project/outputs/` rather than rerunning the expensive backtests.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path('/Users/scottthomasswitzer/Desktop/FIN496FP/FIN496-Foundation-Project')
OUTPUT_DIR = Path('/Users/scottthomasswitzer/Desktop/FIN496FP/FIN496-Foundation-Project/taa_project/outputs/runs/timesfm_vb10/outputs')
plt.style.use('default')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


## Data Profile

In [ ]:
inceptions = pd.read_csv(OUTPUT_DIR / 'asset_inception_dates.csv')
gap_summary = pd.read_csv(OUTPUT_DIR / 'asset_gap_summary.csv')
gap_detail = pd.read_csv(OUTPUT_DIR / 'asset_gap_detail.csv')
display(inceptions)
display(gap_summary.head(15))


In [ ]:
if not gap_detail.empty:
    plt.figure(figsize=(8, 4))
    gap_detail['missing_calendar_days'].hist(bins=30)
    plt.title('Gap Histogram')
    plt.xlabel('Missing Calendar Days')
    plt.ylabel('Count')
    plt.show()
else:
    print('No gap detail rows to plot.')


## SAA Method Comparison

In [ ]:
saa_methods = pd.read_csv(OUTPUT_DIR / 'saa_method_comparison.csv')
display(saa_methods)
plt.figure(figsize=(8, 4))
plt.bar(saa_methods['method'], saa_methods['sharpe'])
plt.title('SAA Method Sharpe Comparison')
plt.xticks(rotation=20)
plt.show()


## Signal Diagnostics

In [ ]:
oos_regimes = pd.read_csv(OUTPUT_DIR / 'oos_regimes.csv', parse_dates=['date'])
display(oos_regimes['regime'].value_counts(dropna=False).rename_axis('regime').reset_index(name='count'))


In [ ]:
from taa_project.data_loader import load_prices
from taa_project.signals.trend_faber import trend_score
from taa_project.signals.momentum_adm import adm_score, cross_sectional_rank
from taa_project.backtest.walkforward import SLEEVE_BUCKETS
prices = load_prices()
trend = trend_score(prices)
future_21d = np.log(prices).diff(21).shift(-21)
trend_hits = ((trend * future_21d).reindex(columns=trend.columns) > 0).mean().sort_values(ascending=False)
display(trend_hits.rename('trend_hit_rate').to_frame())
adm = cross_sectional_rank(adm_score(prices), SLEEVE_BUCKETS)
aligned_future = future_21d.reindex(adm.index)
ics = []
for dt in adm.index:
    x = adm.loc[dt]
    y = aligned_future.loc[dt]
    valid = x.notna() & y.notna()
    if valid.sum() >= 3:
        ics.append(x[valid].corr(y[valid], method='spearman'))
print('ADM mean IC:', float(pd.Series(ics).mean()) if ics else 'n/a')


In [ ]:
dsr_summary = pd.read_csv(OUTPUT_DIR / 'dsr_summary.csv')
if int(dsr_summary.loc[0, 'timesfm_enabled']) == 0:
    print('TimesFM diagnostics skipped: baseline run used --no-timesfm.')
else:
    print('TimesFM was enabled; inspect saved forecast cache if present.')


## Walk-Forward Validation

In [ ]:
folds = pd.read_csv(OUTPUT_DIR / 'walkforward_folds.csv', parse_dates=['train_start','train_end','embargo_start','embargo_end','test_start','test_end'])
per_fold = pd.read_csv(OUTPUT_DIR / 'per_fold_metrics.csv')
display(folds)
display(per_fold)


## Attribution Decomposition

In [ ]:
attr_saa = pd.read_csv(OUTPUT_DIR / 'attribution_saa_vs_bm2.csv')
attr_taa = pd.read_csv(OUTPUT_DIR / 'attribution_taa_vs_saa.csv')
attr_signal = pd.read_csv(OUTPUT_DIR / 'attribution_per_signal.csv')
display(attr_saa.head(20))
display(attr_taa.head(20))
display(attr_signal)
plot_df = attr_signal[attr_signal['layer'] != 'baseline']
plt.figure(figsize=(8, 4))
plt.bar(plot_df['layer'], plot_df['marginal_oos_sharpe'])
plt.title('Per-Signal Marginal OOS Sharpe')
plt.show()


## Turnover and Cost Profile

In [ ]:
metrics = pd.read_csv(OUTPUT_DIR / 'portfolio_metrics.csv')
display(metrics[['portfolio','sharpe_rf_2pct','sortino_rf_2pct','calmar','turnover_pa','cost_drag_pa','hit_rate']].rename(columns={'sharpe_rf_2pct': 'Sharpe', 'sortino_rf_2pct': 'Sortino', 'calmar': 'Calmar'}))


## Trial Ledger

In [ ]:
trial_ledger = pd.read_csv(Path('/Users/scottthomasswitzer/Desktop/FIN496FP/FIN496-Foundation-Project/TRIAL_LEDGER.csv'))
display(trial_ledger)
display(pd.read_csv(OUTPUT_DIR / 'dsr_summary.csv'))
